In [1]:
!pip install torch tokenizers tqdm

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text_file, seq_length=128):
        with open("alice.txt", 'r', encoding="utf-8") as f:
            text = f.read()
        words = text.split()

        unique_words = sorted(set(words))
        self.word_to_idx = {word: i for i, word in enumerate(unique_words)}
        self.idx_to_word = {i: word for i, word in enumerate(unique_words)}
        self.vocab_size = len(unique_words)

        self.data = [self.word_to_idx[word] for word in words]
        self.seq_length = seq_length
    def __len__(self):
        return len(self.data) - self.seq_length
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_length])
        y = torch.tensor(self.data[idx+1:idx+self.seq_length+1])
        return x, y

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Embedding(5000, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoder(positions)
        x = x.transpose(0, 1)
        x = self.transformer(x)
        x = x.transpose(0, 1)
        return self.fc(x) # Transformer expects (seq_len, batch_size, d_model)

In [ ]:
from tqdm import tqdm

dataset = TextDataset('alice.txt')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = TransformerLM(vocab_size=dataset.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch}')
    # for epoch in range(10):
    for x, y in dataloader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

c:\Users\mason\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epoch 0:   0%|          | 0/920 [06:12<?, ?it/s, loss=0.1463]


KeyboardInterrupt: 

# Adjustments to run quicker:

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import os

In [4]:
class TextDataset(Dataset):
    def __init__(self, text_file, seq_length=128):
        with open(text_file, 'r', encoding='utf-8') as f:
            text = f.read()

        # Byte Pair Encoding Tokenizer forr Subword Tokenization
        if not os.path.exists("tokenizer.json"):
            tokenizer = Tokenizer(models.BPE())
            tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

            trainer = trainers.BpeTrainer(special_tokens=["<PAD>", "<UNK>"], vocab_size=8000)
            tokenizer.train_from_iterator([text], trainer)
            tokenizer.save("tokenizer.json")

        else:
            tokenizer = Tokenizer.from_file("tokenizer.json")

        # Encode text
        encoded = tokenizer.encode(text)
        self.tokenizer = tokenizer
        self.input_ids = encoded.ids
        self.vocab_size = tokenizer.get_vocab_size()

        self.seq_length = seq_length

    def __len__(self):
        return len(self.input_ids) - self.seq_length

    def __getitem__(self, idx):
        x = torch.tensor(self.input_ids[idx:idx+self.seq_length])
        y = torch.tensor(self.input_ids[idx+1:idx+self.seq_length+1])
        return x, y

In [5]:
import torch.nn as nn

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, n_heads=4, num_layers=2, max_seq_len=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_len, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        x = self.embedding(x) + self.positional_encoding[:, :seq_len, :]
        x = x.transpose(0, 1)  # Transformer expects [seq_len, batch, dim]
        out = self.transformer(x)
        out = out.transpose(0, 1)  # Back to [batch, seq_len, dim]
        logits = self.fc_out(out)
        return logits


In [8]:
from tqdm import tqdm

def train_model(text_file='alice.txt', epochs=1, seq_length=64, batch_size=2, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    dataset = TextDataset(text_file, seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TransformerLM(vocab_size=dataset.vocab_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"Vocab size: {dataset.vocab_size}")

    for epoch in range(epochs):
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        total_loss = 0.0

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            output = model(x)
            loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}\n")

    return model, dataset

# Baseline
Note: Embed_dim = 128, n_heads = 4, and num_layers = 2

In [9]:
if __name__ == "__main__":
    model, dataset = train_model('alice.txt', epochs=1, seq_length=64, batch_size=512, lr=1e-3)


Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Vocab size: 4000


Epoch 1/1: 100%|██████████| 78/78 [00:13<00:00,  5.84it/s, loss=4.3375]

Epoch 1 complete — avg loss: 5.4754



# Testing various variations:

This should measure how different configurations of embedding dimensions, n_heads, and the number of layers can influence the performance of a transformer.

This comparison training loop is similar to the previous one with the baseline but has adjustable hyperparameters for adjusting to see how different configurations of transformers effect outcomes.

In [10]:
def comparison_train_model(text_file='alice.txt', epochs=1, seq_length=64, batch_size=512, lr=1e-3, embed_dim=128, n_heads=4, num_layers=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")


    dataset = TextDataset(text_file, seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TransformerLM(
        vocab_size=dataset.vocab_size,
        embed_dim=embed_dim,
        n_heads=n_heads,
        num_layers=num_layers
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"Vocab size: {dataset.vocab_size}")
    print(f"Model params: {sum(p.numel() for p in model.parameters())}")

    model.train()
    for epoch in range(epochs):
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        total_loss = 0.0

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            output = model(x)
            loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}")

    return model, avg_loss

This is establishing the actual comparison loop with a few hand picked examples for comparison.

Again, our baseline hyperparameters were: num_layers=2, n_heads=4, embed_dim=128.

Configs showing decreases in hyperparameters:

In [11]:
configs_dec = [
    # Configs depicting single variable decreases:

    {"embed_dim": 128,  "n_heads": 4, "num_layers": 1},
    {"embed_dim": 128,  "n_heads": 1, "num_layers": 2},
    {"embed_dim": 32,  "n_heads": 4, "num_layers": 2},

    # Config depicting multiple variable decreases:

    {"embed_dim": 32, "n_heads": 1, "num_layers": 1},
]

results_dec = []

for cfg in configs_dec:
    print(f"\nTraining config: {cfg}")
    model, dataset = comparison_train_model(
        text_file='alice.txt',
        epochs=1,
        seq_length=64,
        batch_size=512,
        lr=1e-3
    )

    num_params = sum(p.numel() for p in model.parameters())
    results_dec.append({"config": cfg, "params": num_params})


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Training config: {'embed_dim': 128, 'n_heads': 4, 'num_layers': 1}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:12<00:00,  6.26it/s, loss=4.3658]


Epoch 1 complete — avg loss: 5.4714

Training config: {'embed_dim': 128, 'n_heads': 1, 'num_layers': 2}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:12<00:00,  6.23it/s, loss=4.2830]


Epoch 1 complete — avg loss: 5.4804

Training config: {'embed_dim': 32, 'n_heads': 4, 'num_layers': 2}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:12<00:00,  6.25it/s, loss=4.3543]


Epoch 1 complete — avg loss: 5.4696

Training config: {'embed_dim': 32, 'n_heads': 1, 'num_layers': 1}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:13<00:00,  5.98it/s, loss=4.2586]

Epoch 1 complete — avg loss: 5.4717


These showed similar results implying something was wrong potentially.

In [12]:

configs_in = [
    # Configs depicting single variable increases:

    {"embed_dim": 128,  "n_heads": 4, "num_layers": 6},
    {"embed_dim": 128,  "n_heads": 16, "num_layers": 2},
    {"embed_dim": 512,  "n_heads": 4, "num_layers": 2},

    # Config depicting multiple variable increases:

    {"embed_dim": 512, "n_heads": 16, "num_layers": 6},
]

results_inc = []

for cfg in configs_in:
    print(f"\nTraining config: {cfg}")
    model, dataset = comparison_train_model(
        text_file='alice.txt',
        epochs=1,
        seq_length=64,
        batch_size=512,
        lr=1e-3
    )

    num_params = sum(p.numel() for p in model.parameters())
    results_inc.append({"config": cfg, "params": num_params})



Training config: {'embed_dim': 128, 'n_heads': 4, 'num_layers': 6}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:12<00:00,  6.16it/s, loss=4.2087]


Epoch 1 complete — avg loss: 5.4720

Training config: {'embed_dim': 128, 'n_heads': 16, 'num_layers': 2}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:13<00:00,  5.97it/s, loss=4.3547]


Epoch 1 complete — avg loss: 5.4895

Training config: {'embed_dim': 512, 'n_heads': 4, 'num_layers': 2}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:12<00:00,  6.09it/s, loss=4.2572]


Epoch 1 complete — avg loss: 5.4874

Training config: {'embed_dim': 512, 'n_heads': 16, 'num_layers': 6}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/1: 100%|██████████| 78/78 [00:13<00:00,  5.98it/s, loss=4.3162]

Epoch 1 complete — avg loss: 5.4861


These were also ran and they seemed to have problems as well.

These previous methods included multiple single variable changes, and caused problems and didn't show differences with training.

So, I changed batch size, epochs, and decided to train the baseline again, as well as all variables increasing, and all variables decreasing to see clearly what will happen with more effort given to each.

In [14]:

configs_in = [
    # Config with baseline values:

    {"embed_dim": 128, "n_heads": 4, "num_layers": 2},

    # Config depicting multiple variable decreases:

    {"embed_dim": 32, "n_heads": 1, "num_layers": 1},

    # Config depicting multiple variable increases:

    {"embed_dim": 512, "n_heads": 16, "num_layers": 6},
]

results = []

for cfg in configs_in:
    print(f"\nTraining config: {cfg}")
    model, dataset = comparison_train_model(
        text_file='alice.txt',
        epochs=5,
        seq_length=64,
        batch_size=16,
        lr=1e-3
    )

    num_params = sum(p.numel() for p in model.parameters())
    results.append({"config": cfg, "params": num_params})



Training config: {'embed_dim': 128, 'n_heads': 4, 'num_layers': 2}

Using device: cuda
Vocab size: 4000
Model params: 2246816


Epoch 1/5: 100%|██████████| 2477/2477 [00:23<00:00, 107.38it/s, loss=0.9450]


Epoch 1 complete — avg loss: 2.0601


Epoch 2/5: 100%|██████████| 2477/2477 [00:23<00:00, 104.79it/s, loss=0.7387]


Epoch 2 complete — avg loss: 0.8085


...

Epoch 3 complete — avg loss: 0.5675


Epoch 4/5: 100%|██████████| 2477/2477 [00:23<00:00, 106.91it/s, loss=0.1769]


Epoch 4 complete — avg loss: 0.2939


Epoch 5/5: 100%|██████████| 2477/2477 [00:23<00:00, 105.49it/s, loss=0.1571]


Epoch 5 complete — avg loss: 0.1926


These are the notable results that show the differences. The big thing to keep in mind here is relativety. This is a small dataset and small test and small transformers in general. So of course they will perform similarly, is my guess. However, the key is to look at how they compare.

In the README.md, these results (avg loss) are what I refer to (^). For instance, if I say that the complex (third) run performs 15% better than the simple (second) run, I am comparing 0.2939 (Epoch 4 avg loss complex version) to the number 0.3437 (Epoch 4 avg loss simple version). Then, ( 0.3437 - 0.2939 ) / (0.3437) = ~ 15%